In [2]:
# Single-cell Chatbot using TensorFlow in Jupyter Notebook

import sys
import subprocess
import importlib.util
import json
import random
import re
import numpy as np

# ---------- Auto-install required packages ----------
required_packages = {
    "numpy": "numpy",
    "nltk": "nltk",
    "tensorflow": "tensorflow"
}

for import_name, pip_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

# ---------- Download NLTK data ----------
import nltk

resources = ["punkt", "punkt_tab", "wordnet", "omw-1.4"]

for r in resources:
    try:
        nltk.data.find(r)
    except LookupError:
        nltk.download(r)

lemmatizer = WordNetLemmatizer()

# ---------- Intents data ----------
# You can add more intents, patterns, and responses here
intents = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["Hi", "Hello", "Hey", "Good morning", "Good evening"],
            "responses": ["Hello!", "Hi there!", "Hey! How can I help you?"]
        },
        {
            "tag": "goodbye",
            "patterns": ["Bye", "See you later", "Goodbye", "Take care"],
            "responses": ["Goodbye!", "See you later!", "Take care!"]
        },
        {
            "tag": "thanks",
            "patterns": ["Thanks", "Thank you", "That's helpful", "Thanks a lot"],
            "responses": ["You're welcome!", "Glad I could help!", "Anytime!"]
        },
        {
            "tag": "name",
            "patterns": ["What is your name?", "Who are you?", "Tell me your name"],
            "responses": ["I am your TensorFlow chatbot.", "You can call me TensorBot."]
        },
        {
            "tag": "nlp",
            "patterns": ["What is NLP?", "Explain NLP", "Tell me about natural language processing"],
            "responses": [
                "NLP stands for Natural Language Processing. It helps computers understand and process human language.",
                "Natural Language Processing is a field of AI that works with text and speech."
            ]
        },
        {
            "tag": "python",
            "patterns": ["What is Python?", "Explain Python", "Tell me about Python programming"],
            "responses": [
                "Python is a popular high-level programming language known for simplicity and readability.",
                "Python is widely used in web development, AI, data science, and automation."
            ]
        }
    ]
}

# ---------- Prepare training data ----------
words = []
classes = []
documents = []
ignore_letters = ["?", "!", ".", ",", "'"]

for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)
        documents.append((word_list, intent["tag"]))
        if intent["tag"] not in classes:
            classes.append(intent["tag"])

words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in ignore_letters]
words = sorted(set(words))
classes = sorted(set(classes))

training = []
output_empty = [0] * len(classes)

for document in documents:
    bag = []
    word_patterns = [lemmatizer.lemmatize(word.lower()) for word in document[0]]

    for word in words:
        bag.append(1 if word in word_patterns else 0)

    output_row = list(output_empty)
    output_row[classes.index(document[1])] = 1
    training.append([bag, output_row])

random.shuffle(training)
training = np.array(training, dtype=object)

train_x = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]))

# ---------- Build model ----------
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation="softmax"))

sgd = SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
model.compile(loss="categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])

# ---------- Train model ----------
history = model.fit(train_x, train_y, epochs=200, batch_size=5, verbose=0)

print("✅ Chatbot model trained successfully.")
print("Classes:", classes)

# ---------- Helper functions ----------
def clean_up_sentence(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words

def bag_of_words(sentence):
    sentence_words = clean_up_sentence(sentence)
    bag = [0] * len(words)
    for sw in sentence_words:
        for i, word in enumerate(words):
            if word == sw:
                bag[i] = 1
    return np.array(bag)

def predict_class(sentence):
    bow = bag_of_words(sentence)
    res = model.predict(np.array([bow]), verbose=0)[0]
    ERROR_THRESHOLD = 0.25
    results = [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)

    return_list = []
    for r in results:
        return_list.append({
            "intent": classes[r[0]],
            "probability": str(r[1])
        })
    return return_list

def get_response(predicted_intents, intents_json):
    if predicted_intents:
        tag = predicted_intents[0]["intent"]
        for intent in intents_json["intents"]:
            if intent["tag"] == tag:
                return random.choice(intent["responses"])
    return "Sorry, I didn't understand that. Please try again."

def chatbot_reply(message):
    ints = predict_class(message)
    return get_response(ints, intents)

# ---------- Test examples ----------
test_messages = [
    "Hi",
    "What is NLP?",
    "Tell me about Python",
    "Thanks",
    "Bye"
]

print("\n--- Sample Chat ---")
for msg in test_messages:
    print("You :", msg)
    print("Bot :", chatbot_reply(msg))
    print()

# ---------- Optional interactive mode ----------
print("Type 'quit' to stop.\n")
while True:
    message = input("You: ")
    if message.lower() == "quit":
        print("Bot: Goodbye!")
        break
    print("Bot:", chatbot_reply(message))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


✅ Chatbot model trained successfully.
Classes: ['goodbye', 'greeting', 'name', 'nlp', 'python', 'thanks']

--- Sample Chat ---
You : Hi
Bot : Hi there!

You : What is NLP?
Bot : NLP stands for Natural Language Processing. It helps computers understand and process human language.

You : Tell me about Python
Bot : Python is a popular high-level programming language known for simplicity and readability.

You : Thanks
Bot : You're welcome!

You : Bye
Bot : Goodbye!

Type 'quit' to stop.

You: What's UP
Bot: Glad I could help!
You: good
Bot: Hi there!
You: quit
Bot: Goodbye!
